In [ ]:
from google.colab import drive

from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import List, Literal, get_args
from google.colab import userdata
from time import time
import os
import sys
import subprocess

import regex
import json


import pickle

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# @title Drive imports
%cd /content/drive/MyDrive/Szakdoga/Codes
from literature_objects import Segment
import vuvuzela
%cd /content/drive/MyDrive/Szakdoga/Data

/content/drive/MyDrive/Szakdoga/Codes
/content/drive/MyDrive/Szakdoga/Data
BZ

## Get Chapters

In [ ]:
# Put the chapter titles/numbers in a list to chunk the book
import inflect
p = inflect.engine()
#p.number_to_words(i).upper()

starting_phrases = [f"Chapter {i}." for i in range(1, 28)]
#starting_phrases = [f"{i}. jelenet" for i in range(1, 8)] * 2

ending_phrases = [f"Chapter {i}." for i in range(2, 28)] + ["THE END OF BOOK ONE"]
#ending_phrases = ([f"{i}. jelenet" for i in range(1, 8)] * 2)[1:] + ["Üres a szalon. Már csak Einstein hegedűje hallik "]

In [ ]:
# @title Read text

name_of_text_file = "HG1.txt" # @param {type: "string"}
# @markdown By default, the file will be searched for in  "/content/drive/MyDrive/Szakdoga/Data" in the colab environment. cd into a different directory, or provide full path if text file is located elsewhere.

c = 0
test_data: list[Segment] = []
recording: bool = False

with open(name_of_text_file, "r") as f:
    for line in f:
        if len(line) == 0:
            continue
        if starting_phrases[c].lower() in line.lower() and not recording:
            test_data.append(scene := Segment(start_phrase=starting_phrases[c], end_phrase=ending_phrases[c],description=starting_phrases[c]))
            scene.text = line
            recording = True

        if ending_phrases[c].lower() in line.lower() and recording:
            c += 1
            recording = False
            if c >= len(starting_phrases):
                break

            if starting_phrases[c].lower() in line.lower():
                test_data.append(scene := Segment(start_phrase=starting_phrases[c], end_phrase=ending_phrases[c],))
                scene.text = line
                recording = True

        elif recording:
            scene.text += line


## Jelenet darabolás
A gondolat az, hogy ideális lenne nem kézzel jelenetekre bontani a műveket, mert az nagyon sok munka, de szükséges lehet, mert úgy gondolom, hogy jóval hatékonyabb chunkonként feldolgozni egy művet, mint egyben.
A jelenet egy ideális méretű egységnek tűnik, mert tartalmi egységet képez, ám számban jóval alatta van a bekezdéseknek, pláne a mondatoknak.

In [ ]:
def fuzzy_split(text, delimiters, max_errors=1):
    # Create a pattern: (delim1|delim2|delim3){e<=1}
    # {e<=1} allows up to 1 error (insertion, deletion, or substitution)
    pattern = f"({'|'.join(delimiters)}){{e<={max_errors}}}"

    # Use regex.split to break the text
    parts = regex.split(pattern, text)
    joined_results = [parts[0]]
    for i in range(1, len(parts) - 2, 2):
        joined_results.append(parts[i] + " " + parts[i+1])

    joined_results[-1] += parts[-1]
    return joined_results

In [ ]:
#Segment the text for testing. I chose the second and third chapters.
start_phrase = "CHAPTER ONE"
end_phrase = "THE END"

In [ ]:
text = ""
recording = False


with open("HP1.txt", "r") as f:
    for line in f:
        if start_phrase in line:
            recording = True

        if end_phrase in line:
            recording = False
            break

        if recording:
            text += line.lower()


In [ ]:
## Set up the gemini api
model_name = "gemini-3-flash-preview"

## Client authentication
api_key = None
try:
    with open("api_auth.txt", "r") as f:
        api_key = f.readline().strip()
    print("API key loaded from api_auth.txt.")
except FileNotFoundError:
    print("No api_key file")

client = genai.Client(api_key=api_key)

#----------------------------------------------

# Generating resposne format
general_prompt = "In the following text, separate the different scenes from each other. One scene is logically coherent, usually at the same place in time and space, with mostly the same characters"
enlarge_scenes = ""
enlarge_scenes += "Two scenes are treated the same if one scene is only a story within the given scene" #in new line, so you can comment out variable's value without causing eeror in the api call

class Scene(BaseModel):
    start_phrase: str = Field(description="The first sentence of the scene.")
    end_phrase: str = Field(description="The last sentence of the scene.")
    description: str = Field(description="A 3 word description of the scene")

class SceneList(BaseModel):
    scenes: List[Scene] = Field(description="List of the scenes in the story.")



#Generating response
response = client.models.generate_content(
    model=model_name,
    contents=f"{general_prompt + enlarge_scenes} \n {text}",
    config={
        "response_mime_type": "application/json",
        "response_json_schema": SceneList.model_json_schema(),
    },
)

response_dict = json.loads(response.text)["scenes"]

API key loaded from api_auth.txt.


In [ ]:
c = 0
test_list: list[Segment] = []
recording: bool = False

for c, scene_text in enumerate(fuzzy_split(text, [s["start_phrase"] for s in response_dict][1:], 4)):
    print(c, end= "   ")
    test_list.append(scene := Segment(start_phrase=response_dict[c]["start_phrase"], end_phrase=response_dict[c]["end_phrase"], description=response_dict[c]["description"]))
    scene.text = scene_text

0   1   2   3   4   5   6   7   8   9   10   11   12   13   14   15   16   17   18   19   20   21   22   23   24   25   26   27   28   29   30   31   32   33   34   35   36   37   38   39   40   41   42   43   44   45   46   47   48   

In [ ]:
if len(test_list) != len(response_dict):
    print(f"extracted_scenes: {len(test_list)}")
    print(f"response_scenes: {len(response_dict)}")
    raise Exception("The number of scenes does not match the number of scenes in the response. \n Check the last segment correctly appended. \n Possibly issue: LLM generated wrong quote or the generated quote appears multiple times in the text.")

extracted_scenes: 49
response_scenes: 53


Exception: The number of scenes does not match the number of scenes in the response. 
 Check the last segment correctly appended. 
 Possibly issue: LLM generated wrong quote or the generated quote appears multiple times in the text.

## Get model response

In [ ]:
# @title Model setup {run: "auto"}
# @markdown Name of Gemini model
model_name = "gemini-3-flash-preview" # @param ["gemini-3-flash-preview", "gemini-2.0-flash", "gemini-3.1-flash-lite-preview", "gemini-3.1-pro-preview" ]

# @markdown Name of the file containing the prompt
prompt_file_name = "test_prompt.txt" # @param {type: "string"}

try:
    with open(prompt_file_name) as f:
        prompt = f.read()
        print(f"Prompt loaded from {prompt_file_name}")
except FileNotFoundError:
    print("No prompt file")

#---------------------------------------------------------

# @markdown Check this box, if the created network is allowed to have assimetric connections
is_directed = True #@param {type: "boolean"}

if is_directed:
    prompt += "Assimetries are allowed in an interaction's scores."

sentiments = Literal[-2, -1, 0, 1, 2]

#Response formating

class Connection(BaseModel):
    name: str = Field(description="Name of the connection")
    description: sentiments = Field(description=f"Description of the connection with {max(get_args(sentiments))} being very positive, 0 neutral, and {min(get_args(sentiments))} very negative")
    quote: str = Field(description="A characteristic quote from or about the interaction of the two characters") # in an effort to reduce pollution from training knowledge.
    # weight: int = Field(description="An int between 1 and 100 to describe how important this connection is likely to be")

class Character(BaseModel):
    name: str = Field(description="Name of the character")
    connections: List[Connection] = Field(description="List of the names of the connections of the character")

class CharacterList(BaseModel):
    characters: List[Character] = Field(description="List of the characters in the story")

## Client authentication
# @markdown Name of the secret the api key is stored in
api_secret_name = "Gemini-szakdoga" #@param {type: "string"}
try:
    api_key = userdata.get(api_secret_name)
except Exception as e:
    print("Problem with the api key: \n", e)

try:
    client = genai.Client(api_key=api_key)
except Exception as e:
    print("Problem with the api key: \n", e)

No prompt file


NameError: name 'prompt' is not defined

In [ ]:
responses: list[dict] = []

#test_data: list[Segment] = test_list #change this dict to change the test set

for i, scene in enumerate(test_data[5:]):
    print(f"Generating response for scene No. {i+1}/{len(test_data)}", flush=True)

    response = client.models.generate_content(
        model=model_name,
        contents=f"{prompt} \n {scene.text}",
        config=types.GenerateContentConfig(
                safety_settings=[
                    types.SafetySetting(
                        category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
                        threshold=types.HarmBlockThreshold.BLOCK_NONE,
                        ),
                    types.SafetySetting(
                        category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
                        threshold=types.HarmBlockThreshold.BLOCK_NONE,
                        ),
                    types.SafetySetting(
                        category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
                        threshold=types.HarmBlockThreshold.BLOCK_NONE,
                        ),
                    types.SafetySetting(
                        category=types.HarmCategory.HARM_CATEGORY_CIVIC_INTEGRITY,
                        threshold=types.HarmBlockThreshold.BLOCK_NONE,
                        ),
                    types.SafetySetting(
                        category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
                        threshold=types.HarmBlockThreshold.BLOCK_NONE,
                        ),
                    ],
                temperature=0,
                response_mime_type="application/json",
                response_json_schema=CharacterList.model_json_schema(),
            ))
    scene.network = json.loads(response.text)
    responses.append(scene.network)


Generating response for scene No. 1/27
Generating response for scene No. 2/27
Generating response for scene No. 3/27
Generating response for scene No. 4/27
Generating response for scene No. 5/27
Generating response for scene No. 6/27
Generating response for scene No. 7/27
Generating response for scene No. 8/27
Generating response for scene No. 9/27
Generating response for scene No. 10/27
Generating response for scene No. 11/27
Generating response for scene No. 12/27
Generating response for scene No. 13/27
Generating response for scene No. 14/27
Generating response for scene No. 15/27
Generating response for scene No. 16/27
Generating response for scene No. 17/27
Generating response for scene No. 18/27
Generating response for scene No. 19/27
Generating response for scene No. 20/27
Generating response for scene No. 21/27
Generating response for scene No. 22/27


In [ ]:
for i in responses[0]['characters']:
    print(i)

{'name': 'FELÜGYELŐ', 'connections': [{'name': 'Martha főnővér', 'description': -1, 'quote': 'Mondja, én megőrültem?'}, {'name': 'BLOCHER', 'description': 1, 'quote': 'Blocher, fényképezhetsz.'}, {'name': 'GUHL', 'description': 1, 'quote': 'Jegyezte a főnővér vallomását, Guhl?'}, {'name': 'TÖRVÉNYSZÉKI ORVOS', 'description': 0, 'quote': 'Megfojtották, doktor?'}, {'name': 'Ernst Heinrich Ernesti (Einstein)', 'description': -2, 'quote': 'A fickó mégiscsak megfojtott egy ápolónőt!'}, {'name': 'Herbert Georg Beutler (Newton)', 'description': -1, 'quote': 'Ebbe már nem megyek bele, Martha főnővér. Beszélnem kell a főorvosnővel.'}]}
{'name': 'Martha főnővér', 'connections': [{'name': 'FELÜGYELŐ', 'description': -1, 'quote': 'Orvosilag nem engedélyezhetjük. Ernesti úrnak most hegedülnie kell.'}, {'name': 'Ernst Heinrich Ernesti (Einstein)', 'description': 2, 'quote': 'Ez nem fickó, ez egy beteg ember, akinek nyugalomra van szüksége.'}, {'name': 'Irene Straub', 'description': 1, 'quote': 'Iren

## Regularizing names

Problems:
- HP2 Nearly headless Nick is present twice

In [ ]:
names: set = set()

for response in test_data:
    if response.network is None:
        continue
    for character in response.network['characters']:
        names.add(character['name'])

        for connection in character['connections']:
            names.add(connection['name'])

In [ ]:
class Names(BaseModel):
    name: str = Field(description="Full name of the character")
    aliases: List[str] = Field(description="List of the aliases of the character. Include the name in the name field")

class NamesList(BaseModel):
    names: List[Names] = Field(description="List of the names of the characters in the story")

response = client.models.generate_content(
    model=model_name,
    contents=f"In the following list, cluster together the different names for the characters: \n {list(names)}",
     config={
            "response_mime_type": "application/json",
            "response_json_schema": NamesList.model_json_schema(),
        },
    )

name_dict = json.loads(response.text)['names']
name_map = { alias : name['name'] for name in name_dict for alias in name['aliases']}


In [ ]:
for scene in test_data:
    if scene.network is None:
        continue
    for character in scene.network['characters']:
        character["name"] = name_map[character["name"]]

        for connection in character['connections']:
            if connection['name'] in name_map:
                connection['name'] = name_map[connection['name']]

## Saving

In [ ]:


name_of_pickle_file = "HG1_by_chapter_no_chapter5.p" # @param {type: "string"}
pickle.dump(test_data, open(name_of_pickle_file, "wb"))

#The sound of success
# Author: Nepusz Tamás
# File: vuvuzela.py
# License: MIT
%cd ../Codes
process = subprocess.Popen(["python", "vuvuzela.py"], stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True)
end_time = time() + 5

while time() < end_time:
    # Read 1 character at a time so it doesn't wait for a newline (\n)
    char = process.stdout.read(1)
    if char:
        sys.stdout.write(char)
        sys.stdout.flush()

process.terminate()
process.wait()


/content/drive/MyDrive/Szakdoga/Codes
BzzZZZzZzZzzZzzzZzzzzzz

0